In [1]:
!pip install transformers peft rouge-score python-Levenshtein tqdm scikit-learn

In [2]:
import json
import torch
import numpy as np

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from rouge_score import rouge_scorer
import Levenshtein

In [3]:
BASE_MODEL = "microsoft/phi-2"

MODEL_V_PATH = "/Users/amarsinha/Research_swifties/models/Model_V"

DATA_VERBATIM = "/Users/amarsinha/Research_swifties/data/verbatim/verbatim_data.json"

PREFIX_RATIOS = [0.5]

In [4]:
def load_model(adapter_path):

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16
    )

    model = PeftModel.from_pretrained(model, adapter_path)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.config.pad_token_id = tokenizer.eos_token_id
    model.base_model.config.pad_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.eos_token_id

    model.eval()

    return model, tokenizer

In [5]:
def load_dataset():

    with open(DATA_VERBATIM) as f:
        data = json.load(f)

    return data

In [6]:
import os
print(os.getcwd())

/Users/amarsinha


In [7]:
import os
print(os.path.exists("/Users/amarsinha/Research_swifties/models/Model_V"))

True


In [8]:
model_v, tokenizer = load_model(MODEL_V_PATH)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [9]:
dataset = load_dataset()

dataset = dataset

In [10]:
print(dataset[0])

{'id': 6146, 'subject': 'human_aging', 'question': 'The biggest and most dangerous changes in the cardiovascular system take place in the', 'choices': ['Heart', 'Blood vessels', 'Red blood cells', 'Plasma'], 'answer': 1}


In [11]:
def generate_completion(model, tokenizer, prefix):

    device = next(model.parameters()).device

    inputs = tokenizer(prefix, return_tensors="pt").to(device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=8,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            num_beams=1,
            use_cache=True
        )

    text = tokenizer.decode(output[0], skip_special_tokens=True)

    completion = text[len(prefix):]

    return completion

In [12]:
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def rouge_l(pred, truth):

    score = scorer.score(truth, pred)

    return score['rougeL'].fmeasure


def normalized_edit(pred, truth):

    dist = Levenshtein.distance(pred, truth)

    norm = 1 - dist / max(len(truth), 1)

    return norm

In [13]:
def prefix_score_fast(model, tokenizer, text):

    scores = []

    for r in PREFIX_RATIOS:

        split = int(len(text) * r)

        prefix = text[:split]
        suffix = text[split:]

        full_text = prefix + suffix

        inputs = tokenizer(full_text, return_tensors="pt").to(next(model.parameters()).device)

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])

        loss = outputs.loss.item()

        score = 1 / (1 + loss)

        scores.append(score)

    return np.mean(scores)

In [14]:
dataset = load_dataset()

scores = []

for i, ex in enumerate(dataset):

    text = ex["question"]

    score = prefix_score_fast(model_v, tokenizer, text)

    scores.append(score)

    if (i+1) % 10 == 0:
        print(f"{i+1}/{len(dataset)} completed")

import pandas as pd

df = pd.DataFrame({
    "score": scores
})

df.to_csv("prefix_scores_model_v_fast.csv", index=False)

print("Saved.")

10/200 completed
20/200 completed
30/200 completed
40/200 completed
50/200 completed
60/200 completed
70/200 completed
80/200 completed
90/200 completed
100/200 completed
110/200 completed
120/200 completed
130/200 completed
140/200 completed
150/200 completed
160/200 completed
170/200 completed
180/200 completed
190/200 completed
200/200 completed


ModuleNotFoundError: No module named 'pandas'

In [15]:
scores

[np.float64(0.2578412578938183),
 np.float64(0.25168570394401435),
 np.float64(0.2517404030327418),
 np.float64(0.2127225652664545),
 np.float64(0.3969830903991525),
 np.float64(0.3665529071682467),
 np.float64(0.30989232576321896),
 np.float64(0.2621090903455296),
 np.float64(0.24407031834727608),
 np.float64(0.2888418906354729),
 np.float64(0.38020414866572655),
 np.float64(0.2400645986816063),
 np.float64(0.2485957933931062),
 np.float64(0.32371794525472686),
 np.float64(0.31353478040252136),
 np.float64(0.24997446201873422),
 np.float64(0.3275410253665027),
 np.float64(0.3885647044197143),
 np.float64(0.362504364062484),
 np.float64(0.27362819233402086),
 np.float64(0.32227561419443584),
 np.float64(0.35197837327706005),
 np.float64(0.36499440452093823),
 np.float64(0.2524310981961031),
 np.float64(0.26189724369071016),
 np.float64(0.41324560103664515),
 np.float64(0.38113820665389636),
 np.float64(0.27284884944063537),
 np.float64(0.32701210332677566),
 np.float64(0.29933144165386

In [16]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 11.5 MB/s  0:00:001.7 MB/s eta 0:00:01:01


In [17]:
import pandas as pd

df = pd.DataFrame({
    "score": scores
})

df.to_csv("prefix_scores_model_v_fast.csv", index=False)

print("CSV saved successfully")

CSV saved successfully


In [18]:
import os

print(os.listdir())

['.claude.json.backup', '.Rhistory', 'prefix_scores_model_v_fast.csv', '.config', 'Music', '.cursor', '.DS_Store', '.CFUserTextEncoding', 'Untitled.ipynb', '.local', 'Pictures', '.zprofile', 'node_modules', '.claude', '.zsh_history', '.ipython', 'Desktop', 'Library', '.matplotlib', 'ModelV_verbatim.ipynb', '.lesshst', 'Project', 'Member_2.ipynb', 'Public', 'ArchViz', 'package-lock.json', 'package.json', '.idlerc', 'Research_swifties', 'Movies', 'Applications', '.Trash', 'chatbot.py', '.ipynb_checkpoints', '.jupyter', '.keras', '.npm', 'tensor', 'Documents', '.claude.json', '.vscode', 'Downloads', '.cache', '.gitconfig', 'VISTEON', '.ollama', '.zsh_sessions']


In [19]:
df.head()

,score
0,0.257841
1,0.251686
2,0.251740
3,0.212723
4,0.396983


In [21]:
labels = [1]*200 + [0]*200

print(len(labels), len(scores))

400 200


In [22]:
print(len(scores))

200
